# Practice Session 09: Spectral graph analysis

<font size="+2" color="blue">Additional results: three dimensional projection</font>

In [ ]:
# LEAVE AS-IS

import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sparse
import scipy.sparse.linalg as linalg

# Disable warnings about future changes
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# 1. Compute graph Laplacian

In [ ]:
# LEAVE AS-IS

INPUT_GRAPH_FILENAME = "starwars.graphml"

# Read the graph in GraphML format
sw_in = nx.read_graphml(INPUT_GRAPH_FILENAME)

# Re-label the nodes so they use the 'name' as label
sw_relabeled = nx.relabel.relabel_nodes(sw_in, dict(sw_in.nodes(data='name')))

# Convert the graph to undirected
sw = sw_relabeled.to_undirected()

In [ ]:
# LEAVE AS-IS

plt.figure(figsize=(20, 10))
nx.draw_spring(sw, with_labels=True, node_color='lightgreen', edge_color='lightgrey')
plt.show()

In [ ]:
def diagonal_degree_matrix(g: nx.Graph):
    n = g.number_of_nodes()
    d = np.zeros((n, n)) # Create matrix of zeros of size NxN
    for node, i in zip(g.nodes, range(n)): # Iterate over the nodes of the graph
        d[i, i] = g.degree(node) # Set the diagonal of each node to its degree
    return d

def adjacency_matrix(g: nx.Graph):
    n = g.number_of_nodes()
    a = np.zeros((n, n))
    for n1, i in zip(g.nodes, range(n)): # Iterate over all rows
        for n2, j in zip(g.nodes, range(i)): # Iterate over all columns below the diagonal, this will generate a lower-triangular matrix
            a[i, j] = float(g.has_edge(n1, n2))
        a[:, i] = a[i, :] # Copy the lower-triangle computed elements into upper-triangle
    return a

def laplacian(g: nx.Graph):
    return diagonal_degree_matrix(g) - adjacency_matrix(g)

In [ ]:
# LEAVE AS-IS

def test_equality_matrices(A, B):
    
    # Check equality element-wise using dense matrices comparison
    return np.array_equal(A, B.todense())                                               # ASK - I REMOVED TODENSE()

# Test adjacency matrix

A = adjacency_matrix(sw)
Anx = nx.adjacency_matrix(sw)

if test_equality_matrices(A, Anx):
    print("OK - Adjacency matrix correctly generated")
else:
    print("FAIL - Your adjacency matrix is not equal to the one generated by NetworkX")

# Test Laplacian matrix

L = laplacian(sw)
Lnx = nx.laplacian_matrix(sw)
                
if test_equality_matrices(L, Lnx):
    print("OK - Laplacian matrix correctly generated")
else:
    print("FAIL - Your Laplacian matrix is not equal to the one generated by NetworkX")

# 2. Compute eigenvectors to layout a lattice graph

In [ ]:
# LEAVE AS-IS

Ggrid = nx.grid_2d_graph(6,4)

for i in range(4):
    plt.figure(figsize=(10, 3))
    nx.draw_networkx(Ggrid, with_labels=True, node_color='lightblue')
    plt.show()

# 2.1. Obtain the spectrum of the graph

In [ ]:
# LEAVE AS-IS

# Obtain three eigenvalues and eigenvectors
eigenvalues, eigenvectors = linalg.eigsh(laplacian(Ggrid), k=3, which='SM')

#  The three eigenvectors
first_eigenvector = eigenvectors[:,0]
X_positions = eigenvectors[:,1]
Y_positions = eigenvectors[:,2]

# The first eigenvector should be constant
print("First eigenvector (should be constant):")
print(first_eigenvector)
print()

# Print coordinates
print("Second (X positions) and third (Y positions) eigenvector:")
print(X_positions)
print(Y_positions)

In [ ]:
def spectral_projection(graph: nx.Graph, node_color='lightblue'):
    # Obtain positions X, Y
    eigenvalues, eigenvectors = linalg.eigsh(laplacian(graph), k=3, which='SM')
    X_positions = eigenvectors[:, 1]
    Y_positions = eigenvectors[:, 2]

    return X_positions, Y_positions
    
def draw_graph_fixed_positions(graph: nx.Graph, X_positions, Y_positions, title, xlabel, ylabel,
                               width=20, height=6, 
                               font_color='white', node_color='black'):
    
    # Create the figure of the given width and height,
    # then add title, xlabel, ylabel
    plt.figure(figsize=(width, height))
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    
    # Create the dictionary with positions
    pos = {}
    for node, i in zip(graph.nodes, range(graph.number_of_nodes())):
        pos[node] = (X_positions[i], Y_positions[i])
    
    # Draw graph
    _ = nx.draw_networkx(graph, pos=pos, with_labels=True, node_color=node_color, font_color=font_color)

In [ ]:
# LEAVE AS-IS

def draw_spectral_projection(g, width=20, height=6, font_color='white', node_color='black'):
    X, Y = spectral_projection(g)
    _ = draw_graph_fixed_positions(g, X, Y, node_color=node_color, font_color=font_color,
                               width=width, height=height,
                               title="Spectral projection of graph", 
                               xlabel="First eigenvector", ylabel="Second eigenvector")
    
draw_spectral_projection(Ggrid, font_color='black', node_color='lightblue')

<font size="+1" color="red">We can see that now we get a similar result as before, but now the placement of the nodes in this grid form is always consistent. </font>

# 3. Spectral graph clustering

## 3.1. Clustering of graph generated using the stochastic block model

In [ ]:
# LEAVE AS-IS

n1 = 8
n2 = 12

P = [
    [0.80, 0.07], # Connecting community 1 to itself, or community 1 to community 2
    [0.07, 0.95]  # Connecting community 2 to community 1, or community 2 to itself
] 
Gblock = nx.stochastic_block_model([n1, n2], P)
assert nx.is_connected(Gblock), "Repeat: generated graph is not connected"

# Draw graph
plt.figure(figsize=(20, 8))
color_vec = ["red" for i in range(n1)] + ["blue" for i in range(n2)] 
nx.draw_networkx(Gblock, with_labels=True, node_color=color_vec, font_color='white')


In [ ]:
draw_spectral_projection(Gblock, node_color=color_vec)
plt.axvline(0)
plt.show()

<font size="+1" color="red">We can see that this way of drawing the graph can be used to make minimal cuts in graphs with two communities, since the eigenvector with second smallest eigenvalue will be 0 when the communities are separated. Then we only need to look where is the axis in that eigenvector since where it is equal to 0 then it will be the best cut to separate those communities.</font>

In [ ]:
# LEAVE AS-IS

def generate_connected_block_model_graph(n1, n2, P):
    while True:
        graph = nx.stochastic_block_model([n1, n2], P)

        if nx.is_connected(graph):
            print("Success -- generated graph is connected")
            break
        else:
            print("Not connected, generating again")
            
    return graph

def draw_block_model_graph_and_projection(graph, n1, n2, width=20, height=6,
                                          color1="red", color2="blue", font_color="white"):
    # Create figure
    plt.figure(figsize=(20, 8))
    
    # Create list of colors
    color_vec = [color1 for i in range(n1)] + [color2 for i in range(n2)] 
    
    # Draw spectral projections
    draw_spectral_projection(graph, width=width, height=height, node_color=color_vec, font_color=font_color)
    
    # Add a vertical line at x=0
    _ = plt.axvline(0.0, color='blue')

In [ ]:
P1 = [
    [0.80, 0.45], # Connecting community 1 to itself, or community 1 to community 2
    [0.45, 0.95]  # Connecting community 2 to community 1, or community 2 to itself
] 
Gblock1 = generate_connected_block_model_graph(n1, n2, P1)
draw_block_model_graph_and_projection(Gblock1, n1, n2, width=20, height=8)

In [ ]:
P1 = [
    [0.80, 0.6], # Connecting community 1 to itself, or community 1 to community 2
    [0.6, 0.95]  # Connecting community 2 to community 1, or community 2 to itself
] 
Gblock1 = generate_connected_block_model_graph(n1, n2, P1)
draw_block_model_graph_and_projection(Gblock1, n1, n2, width=20, height=8)

<font size="+1" color="red">Replace this cell with a brief commentary on the graphs you have generated.</font>

## 3.3. Clustering of the Game of Thrones graph

In [ ]:
# LEAVE AS-IS

# Load graph
# (If there is an error of 'long' attribute data type,
#  you can change 'long' to 'string' in the "got.graphml" file.)
got = nx.read_graphml("got.graphml")

# Make undirected, removing multi-edges
got = nx.Graph(got)

# Relabel
got = nx.relabel_nodes(got, nx.get_node_attributes(got, 'name'))


In [ ]:
# LEAVE AS-IS

def extract_by_attribute_values(graph, attribute_name, attribute_values):
    
    nodes_to_keep = []
    
    for node, value in nx.get_node_attributes(graph, attribute_name).items():
        if value in attribute_values:
            nodes_to_keep.append(node)
            
    return nodes_to_keep

In [ ]:
got_selected = got.subgraph(extract_by_attribute_values(got, attribute_name='house', attribute_values=['House Stark', 'House Targaryen']))
got_selected.number_of_nodes()

In [ ]:
# LEAVE AS-IS

colors = []
house_of = nx.get_node_attributes(got_selected, 'house')

node_sequence = list(got_selected.nodes())
for i in range(len(node_sequence)):
    node = node_sequence[i]
    if house_of[node] == 'House Stark':
        colors.append('orange')
    elif house_of[node] == 'House Targaryen':
        colors.append('lightgreen')
    else:
        assert False

In [ ]:
draw_spectral_projection(got_selected, node_color=colors, font_color='Black')
plt.axvline(0)
plt.show()

<font size="+1" color="red">Replace this cell with a brief commentary about what you see in this projection, and what can you conjecture, based on this graph projection, about the interactions between these two houses in the series. Note that the lead character "Jon Snow" can be considered as born in House Targaryen or House Stark, depending on whether one follows the books or the TV series.</font>

## 3.4. Clustering the Karate Club graph

Finally, cluster the Karate Club graph. The following cell, which you should leave as-is, loads this graph, and initializes a list of colors for each node.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

# Load the graph and re-label nodes to use attribute "name"
karate = nx.read_graphml("karate.graphml")
karate = nx.relabel_nodes(karate, nx.get_node_attributes(karate, 'name'))

# Ground-truth communities in which the Karate club splitted
karate_communities = {'1': 'A', '2': 'A', '3': 'A', '4': 'A', '5': 'A', '6': 'A',
                      '7': 'A', '8': 'A', '9': 'B', '10': 'B', '11': 'A', '12': 'A',
                      '13': 'A', '14': 'A', '15': 'B', '16': 'B', '17': 'A', '18': 'A',
                      '19': 'B', '20': 'A', '21': 'B', '22': 'A', '23': 'B', '24': 'B',
                      '25': 'B', '26': 'B', '27': 'B', '28': 'B', '29': 'B', '30': 'B',
                      '31': 'B', '32': 'B', '33': 'B', '34': 'B' }

# Colors
communities_to_colors = {'A': 'red', 'B': 'blue'} 
karate_colors = [communities_to_colors[karate_communities[node]] for node in karate.nodes()]


In [ ]:
draw_spectral_projection(karate, node_color=karate_colors, font_color='Black')
plt.axvline(0)
plt.show()

<font size="+1" color="red">Replace this cell with a brief commentary about what you see in this projection. Are there any misplaced node or nodes? Why do you think that node or those nodes appear in the wrong community? What does it mean that some nodes overlap perfectly in this projection?</font>

# Extra section

In [ ]:
def spectral_projection_3d(graph: nx.Graph, node_color='lightblue'):
    # Obtain positions X, Y, Z
    eigenvalues, eigenvectors = linalg.eigsh(laplacian(graph), k=4, which='SM')
    X_positions = eigenvectors[:, 1]
    Y_positions = eigenvectors[:, 2]
    Z_positions = eigenvectors[:, 3]

    return X_positions, Y_positions, Z_positions
    
def draw_graph_fixed_positions_3d(graph: nx.Graph, X_positions, Y_positions, Z_positions, title, xlabel, ylabel, zlabel,
                               width=20, height=6, 
                               font_color='white', node_color='black'):
    
    # Create the figure of the given width and height,
    # then add title, xlabel, ylabel
    fig = plt.figure(figsize=(width, height))
    ax = fig.add_subplot(111, projection="3d")
    ax.set_title(title, fontsize=18)
    ax.set_box_aspect(aspect=None, zoom=0.9)
    
    ax.grid(False)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_zlabel(zlabel, fontsize=12)
    
    # Create the dictionary with positions
    pos = {}
    for node, i in zip(graph.nodes, range(graph.number_of_nodes())):
        pos[node] = (X_positions[i], Y_positions[i], Z_positions[i])

    # y = z = np.arange(-0.3, 0.3, 0.1)
    # yy, zz = np.meshgrid(y, z)
    # xx = yy*0
    # ax.plot_surface(xx, yy, zz, color='red', alpha=0.7)
    
    # Draw graph
    node_xyz = np.array([pos[v] for v in sorted(graph)])
    edge_xyz = np.array([(pos[u], pos[v]) for u, v in graph.edges()])

    # Plot the nodes - alpha is scaled by "depth" automatically
    ax.scatter(*node_xyz.T, s=100, ec="w")

    # Plot the edges
    for vizedge in edge_xyz:
        ax.plot(*vizedge.T, color="tab:gray")


def draw_spectral_projection_3d(g, width=40, height=11, font_color='white', node_color='black'):
    X, Y, Z = spectral_projection_3d(g)
    _ = draw_graph_fixed_positions_3d(g, X, Y, Z, node_color=node_color, font_color=font_color,
                               width=width, height=height,
                               title="Spectral 3D projection of graph", 
                               xlabel="First eigenvector", ylabel="Second eigenvector", zlabel="Third eigenvector")

In [ ]:
Ggrid_3d = nx.grid_graph(dim=(1, 5, 10))
draw_spectral_projection_3d(Ggrid_3d, font_color='black', node_color='lightblue')

# Deliver your code (individually)

A .zip file containing:

* This notebook.


## Extra points are available

For extra points and extra learning, create a three-dimensional lattice using networkx and draw its 3D spectral projection. You will need three vectors instead of two, i.e., you will need to extract the eigenvectors of the four smallest eigenvalues. You will also need to create a 3D scatter plot in matplotlib.

**Note:** if for extra points you go, ``<font size="+2" color="blue">Additional results: three dimensional projection</font>`` at the top of your notebook, you must add.

<font size="-1" color="gray">(This cell, when delivering, remove.)</font>

<font size="+2" color="#003300">I hereby declare that, except for the code provided by the course instructors, all of my code, report, and figures were produced by myself.</font>